# Align an LLM and Build RAG with Nugen API

In this tutorial, we'll walk through two key capabilities of the Nugen platform:

1. **Model Alignment** — Fine-tune a small LLM on a domain-specific document using the Nugen Alignment API
2. **RAG (Retrieval-Augmented Generation)** — Build a simple question-answering system that retrieves relevant context from the document and uses the aligned model to generate accurate answers

**What we'll use:**
- A summary of the **Indian Factories Act, 1948** as our domain document
- **Qwen 2.5 0.5B** as the base model for alignment (small and fast)
- **Nomic Embed Text v1.5** for generating text embeddings
- Pure Python with NumPy — no heavy dependencies like vector databases

By the end, you'll have a working pipeline that can answer questions about the Factories Act using your aligned model.

## Prerequisites

You'll need:
- A **Nugen API key** (get one free at [platform.nugen.in](https://platform.nugen.in))
- Python 3.8+ with `requests` and `numpy` installed

In [ ]:
# Install dependencies (skip if already installed)
!pip install requests numpy python-dotenv --quiet

## Step 1: Setup and Authentication

Load your Nugen API key. We recommend storing it in a `.env` file rather than hardcoding it.

Create a `.env` file in this directory:
```
NUGEN_API_KEY=your_api_key_here
```

In [ ]:
import os
import json
import time
import requests
import numpy as np
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("NUGEN_API_KEY")
BASE_URL = "https://api.nugen.in/api/v3"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

# Quick check — make sure the key works
r = requests.get(f"{BASE_URL}/models/base", headers=HEADERS)
if r.status_code == 200:
    models = r.json()["models"]
    print(f"Connected to Nugen API. {len(models)} models available.")
else:
    print(f"Error: {r.status_code} — check your API key")

Let's see which models support alignment (fine-tuning):

In [ ]:
# Show alignment-ready models
for m in models:
    if m["alignment_ready"]:
        print(f"  {m['id']:40s} — {m['description']}")

We'll use **`qwen-v2p5-0p5b-instruct`** — a small 0.5B parameter model. It aligns quickly and is perfect for demonstrating the workflow.

---

## Step 2: Upload a Document

The Nugen alignment API learns from documents you upload. We'll use a summary of the **Indian Factories Act, 1948** — a real piece of Indian labor law covering health, safety, welfare, and working hours in factories.

Supported file types: `.txt`, `.json`, `.jsonl`, `.md`, `.xml`, `.csv`

In [ ]:
# Upload the document
doc_path = "indian_factories_act_summary.txt"

upload_headers = {"Authorization": f"Bearer {API_KEY}"}
files = [("files", (doc_path, open(doc_path, "rb"), "text/plain"))]

r = requests.post(f"{BASE_URL}/documents", headers=upload_headers, files=files)
upload_id = r.json()["document_ids"][0]
print(f"Upload ID: {upload_id}")

In [ ]:
# Wait for the document to finish processing
while True:
    r = requests.get(f"{BASE_URL}/documents/{upload_id}", headers=HEADERS)
    status = r.json()["status"]
    doc_id = r.json()["document_id"]
    print(f"Status: {status}")
    if status == "READY":
        break
    time.sleep(3)

print(f"\nDocument ready. Document ID: {doc_id}")

---

## Step 3: Align the Model

Now we create an alignment project. This tells Nugen to fine-tune `qwen-v2p5-0p5b-instruct` on our document so it becomes a specialist in the Factories Act.

In [ ]:
# Create alignment project
alignment_payload = {
    "name": "factories-act-alignment",
    "base_model": "qwen-v2p5-0p5b-instruct",
    "document_ids": [doc_id],
    "description": "Align Qwen 0.5B on Indian Factories Act for domain-specific QA",
}

r = requests.post(
    f"{BASE_URL}/alignment-project/create",
    headers=HEADERS,
    json=alignment_payload,
)

alignment_id = r.json()["alignment_id"]
print(f"Alignment started!")
print(f"Alignment ID: {alignment_id}")
print(f"Status: {r.json()['status']}")

In [ ]:
# Poll until alignment completes
# This typically takes 5-15 minutes for a small model + small document

print("Waiting for alignment to complete...")
print("(This may take 5-15 minutes)\n")

while True:
    r = requests.get(
        f"{BASE_URL}/alignment-project/status/{alignment_id}",
        headers=HEADERS,
    )
    result = r.json()
    status = result["status"]
    elapsed = ""
    if result.get("start_time"):
        elapsed = f" (started: {result['start_time'][:19]})"

    print(f"  Status: {status}{elapsed}")

    if status in ("READY", "COMPLETED", "FAILED", "STOPPED"):
        break
    time.sleep(30)

print(f"\nAlignment finished with status: {status}")
if result.get("data"):
    aligned_model_id_from_alignment = result["data"].get("model_id")
    print(f"Aligned model ID: {aligned_model_id_from_alignment}")
    print(f"Details: {json.dumps(result['data'], indent=2)}")

---

## Step 4: Deploy the Aligned Model

Once alignment is complete, we need to deploy the aligned model so we can use it for inference.

In [ ]:
# List aligned models to find ours
r = requests.get(f"{BASE_URL}/models/aligned", headers=HEADERS)
aligned_models = r.json()

print("Your aligned models:")
for m in aligned_models:
    print(f"  ID: {m['id']}  |  Status: {m.get('status', 'N/A')}")

# Use the most recently aligned model
aligned_model_id = aligned_models[-1]["id"] if aligned_models else None
print(f"\nUsing aligned model: {aligned_model_id}")

In [ ]:
# Deploy the aligned model
r = requests.post(
    f"{BASE_URL}/models/deploy-model/{aligned_model_id}",
    headers=HEADERS,
)
print(f"Deploy request: {r.status_code}")
print(r.json())

In [ ]:
# Wait for deployment
print("Waiting for model deployment...\n")

while True:
    r = requests.get(
        f"{BASE_URL}/models/deploy-model/{aligned_model_id}/status",
        headers=HEADERS,
    )
    result = r.json()
    print(f"  Deployment status: {result['status']}")
    if result["status"] in ("COMPLETED", "FAILED"):
        break
    time.sleep(15)

print(f"\nModel deployed: {aligned_model_id}")

### Quick test — ask the aligned model a question directly

Before building the full RAG pipeline, let's see how the aligned model responds on its own:

In [ ]:
def chat(model, question, context=None, max_tokens=300):
    """Send a question to a Nugen model and return the answer."""
    if context:
        prompt = (
            f"Use the following context to answer the question.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}"
        )
    else:
        prompt = question

    r = requests.post(
        f"{BASE_URL}/inference/chat/completions",
        headers=HEADERS,
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "temperature": 0.3,
        },
    )
    return r.json()["choices"][0]["message"]["content"]


# Ask the aligned model without RAG context
answer = chat(aligned_model_id, "What is the maximum weekly working hours under the Factories Act?")
print("Aligned model (no context):")
print(answer)

---

## Step 5: Build a RAG Pipeline

Now let's build RAG on top of the aligned model. The pipeline has these stages:

1. **Chunk** the document into smaller pieces (we'll try 3 different methods)
2. **Embed** each chunk using Nugen's embedding model
3. **Enhance** the user's query for better retrieval
4. **Retrieve** the most similar chunks via cosine similarity
5. **Re-rank** the retrieved chunks using Nugen's reranker for precision
6. Pass the top chunks as **context** to the aligned model

### 5.1 — Chunking Methods

Different chunking strategies affect retrieval quality. Let's implement three methods and compare them.

In [ ]:
# Read the document
with open("indian_factories_act_summary.txt", "r") as f:
    full_text = f.read()


# --- Method 1: Paragraph-based chunking ---
# Splits on double newlines. Each paragraph becomes a chunk.
# Simple and preserves natural document structure.

def chunk_by_paragraph(text, min_length=50):
    """Split on paragraph boundaries, skip very short fragments."""
    paragraphs = text.split("\n\n")
    return [p.strip() for p in paragraphs if len(p.strip()) > min_length]


# --- Method 2: Fixed-size chunking with overlap ---
# Splits by character count with overlap for continuity.
# Good for uniformly sized chunks regardless of document structure.

def chunk_fixed_size(text, chunk_size=500, overlap=50):
    """Split into fixed-size character windows with overlap."""
    chunks = []
    paragraphs = text.split("\n\n")
    current = ""

    for para in paragraphs:
        if len(current) + len(para) > chunk_size and current:
            chunks.append(current.strip())
            current = current[-overlap:] + "\n\n" + para
        else:
            current = current + "\n\n" + para if current else para

    if current.strip():
        chunks.append(current.strip())

    return chunks


# --- Method 3: Sentence-based chunking ---
# Groups sentences into chunks of N sentences.
# Finer granularity — better for precise retrieval on specific facts.

def chunk_by_sentences(text, sentences_per_chunk=4, overlap_sentences=1):
    """Split into groups of N sentences with overlap."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.replace("\n", " "))
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    chunks = []
    step = sentences_per_chunk - overlap_sentences
    for i in range(0, len(sentences), step):
        group = sentences[i : i + sentences_per_chunk]
        if group:
            chunks.append(" ".join(group))

    return chunks


# Apply all three methods
paragraph_chunks = chunk_by_paragraph(full_text)
fixed_chunks = chunk_fixed_size(full_text, chunk_size=500, overlap=50)
sentence_chunks = chunk_by_sentences(full_text, sentences_per_chunk=4, overlap_sentences=1)

print(f"Method 1 — Paragraph-based:  {len(paragraph_chunks)} chunks")
print(f"Method 2 — Fixed-size (500):  {len(fixed_chunks)} chunks")
print(f"Method 3 — Sentence-based (4 per chunk):  {len(sentence_chunks)} chunks")

### 5.2 — Compare chunking methods

Let's embed all three chunk sets and see which one retrieves the most relevant content for a sample question.

In [ ]:
EMBED_MODEL = "nomic-embed-text-v1.5"


def get_embeddings(texts):
    """Get embeddings for a list of texts using Nugen's embedding API."""
    r = requests.post(
        f"{BASE_URL}/inference/embeddings",
        headers=HEADERS,
        json={"model": EMBED_MODEL, "input": texts},
    )
    return [item["embedding"] for item in r.json()["data"]]


def search_chunks(query, chunk_list, chunk_matrix, top_k=3):
    """Cosine similarity search across a chunk set."""
    q_emb = np.array(get_embeddings([query])[0])
    sims = chunk_matrix @ q_emb / (
        np.linalg.norm(chunk_matrix, axis=1) * np.linalg.norm(q_emb)
    )
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [
        {"chunk": chunk_list[i], "score": float(sims[i]), "index": int(i)}
        for i in top_idx
    ]


# Embed all three chunk sets
para_matrix = np.array(get_embeddings(paragraph_chunks))
fixed_matrix = np.array(get_embeddings(fixed_chunks))
sent_matrix = np.array(get_embeddings(sentence_chunks))

print(f"Paragraph embeddings:  {para_matrix.shape}")
print(f"Fixed-size embeddings: {fixed_matrix.shape}")
print(f"Sentence embeddings:   {sent_matrix.shape}")

In [ ]:
# Compare retrieval across chunking methods
compare_q = "What are the rules about child labor in factories?"

print(f"Query: {compare_q}\n")
print("=" * 70)

for name, c_list, c_matrix in [
    ("Paragraph-based", paragraph_chunks, para_matrix),
    ("Fixed-size (500 chars)", fixed_chunks, fixed_matrix),
    ("Sentence-based (4 per chunk)", sentence_chunks, sent_matrix),
]:
    results = search_chunks(compare_q, c_list, c_matrix, top_k=1)
    best = results[0]
    print(f"\n{name} — top match (score: {best['score']:.3f}):")
    print(f"  {best['chunk'][:150]}...")
    print("-" * 70)

We'll use **paragraph-based chunking** for the rest of this tutorial — it preserves the document's natural structure and works well for legal text where each section covers a distinct topic.

You can swap in `fixed_chunks`/`fixed_matrix` or `sentence_chunks`/`sent_matrix` to experiment with the other methods.

### 5.3 — Query Enhancement

Generic or vague questions can hurt retrieval. We can use an LLM to rewrite the user's question into a more specific, search-friendly version before embedding it.

In [ ]:
def enhance_query(question):
    """Use an LLM to rewrite a vague question into a more specific one."""
    r = requests.post(
        f"{BASE_URL}/inference/chat/completions",
        headers=HEADERS,
        json={
            "model": "llama-v3p2-3b-reasoning",
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "Rewrite this question to be more specific and detailed "
                        "for searching a legal document about Indian factory regulations. "
                        "Return ONLY the improved question, nothing else.\n\n"
                        f"Original question: {question}"
                    ),
                }
            ],
            "max_tokens": 100,
            "temperature": 0.3,
        },
    )
    return r.json()["choices"][0]["message"]["content"]


# Test it
original = "What about kids working?"
enhanced = enhance_query(original)
print(f"Original:  {original}")
print(f"Enhanced:  {enhanced}")

### 5.4 — Re-ranking

After retrieving candidate chunks by cosine similarity, we use Nugen's **reranker** endpoint to re-score them. The reranker is more accurate than embedding similarity alone because it looks at the query and chunk together.

In [ ]:
def rerank_chunks(query, chunks_with_scores, top_k=2):
    """Re-rank retrieved chunks using Nugen's reranker for better precision."""
    texts = [item["chunk"] for item in chunks_with_scores]

    r = requests.post(
        f"{BASE_URL}/inference/reranker",
        headers=HEADERS,
        json={
            "model": aligned_model_id,
            "query": query,
            "texts": texts,
            "top_k": top_k,
        },
    )
    reranked = r.json()["results"]

    return [
        {
            "chunk": texts[item["index"]],
            "rerank_score": item["relevance_score"],
            "original_index": chunks_with_scores[item["index"]]["index"],
        }
        for item in reranked
    ]


# Test reranker
test_q = "What are the rules about child labor?"
candidates = search_chunks(test_q, paragraph_chunks, para_matrix, top_k=5)
reranked = rerank_chunks(test_q, candidates, top_k=2)

print(f"Query: {test_q}\n")
for r in reranked:
    print(f"Chunk {r['original_index']+1} (rerank score: {r['rerank_score']:.3f}):")
    print(f"  {r['chunk'][:120]}...\n")

### 5.5 — Full RAG Pipeline: Enhance + Retrieve + Rerank + Generate

Now we wire everything together into a single `ask` function:

In [ ]:
def ask(question, top_k=2, use_enhancement=True, use_reranker=True):
    """Full RAG pipeline: enhance query -> retrieve -> rerank -> generate."""
    print(f"Original question: {question}")

    # Step 1: Query enhancement
    search_query = question
    if use_enhancement:
        search_query = enhance_query(question)
        print(f"Enhanced query:    {search_query}")

    # Step 2: Retrieve candidates via cosine similarity
    candidates = search_chunks(search_query, paragraph_chunks, para_matrix, top_k=5)

    # Step 3: Rerank for precision
    if use_reranker:
        top_results = rerank_chunks(search_query, candidates, top_k=top_k)
        context = "\n\n".join([r["chunk"] for r in top_results])
        sources = [
            f"Chunk {r['original_index']+1} (rerank: {r['rerank_score']:.3f})"
            for r in top_results
        ]
    else:
        top_results = candidates[:top_k]
        context = "\n\n".join([r["chunk"] for r in top_results])
        sources = [
            f"Chunk {r['index']+1} (sim: {r['score']:.3f})"
            for r in top_results
        ]

    # Step 4: Generate answer using the aligned model
    answer = chat(aligned_model_id, question, context=context)

    return {"question": question, "answer": answer, "sources": sources}

---

## Step 6: Try It Out!

Let's ask some questions about the Indian Factories Act:

In [ ]:
questions = [
    "What is the maximum number of hours a worker can work per week?",
    "What facilities must a factory provide for women workers?",
    "What are the penalties for violating the Factories Act?",
    "At what age is child labor prohibited in factories?",
    "What safety measures are required for dangerous machinery?",
]

for q in questions:
    result = ask(q)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Sources: {', '.join(result['sources'])}")
    print("-" * 70)

---

## Step 7: Compare — Aligned Model vs Base Model

Let's see if the aligned model performs better than the base model with the same RAG context:

In [ ]:
BASE_MODEL = "llama-v3p2-3b-reasoning"
test_q = "What welfare facilities must a factory with 300 workers provide?"

# Get context using the full pipeline (enhance + retrieve + rerank)
enhanced_q = enhance_query(test_q)
candidates = search_chunks(enhanced_q, paragraph_chunks, para_matrix, top_k=5)
reranked = rerank_chunks(enhanced_q, candidates, top_k=2)
context = "\n\n".join([r["chunk"] for r in reranked])

print(f"Question: {test_q}")
print(f"Enhanced: {enhanced_q}\n")
print("=" * 60)

# Base model + RAG
base_answer = chat(BASE_MODEL, test_q, context=context)
print(f"BASE MODEL ({BASE_MODEL}):")
print(base_answer)

print("\n" + "=" * 60)

# Aligned model + RAG
aligned_answer = chat(aligned_model_id, test_q, context=context)
print(f"ALIGNED MODEL ({aligned_model_id}):")
print(aligned_answer)

---

## Summary

In this tutorial, we covered the full workflow:

| Step | What we did | Nugen API used |
|------|-------------|----------------|
| 1 | Uploaded a domain document | `POST /documents` |
| 2 | Aligned a small LLM on the document | `POST /alignment-project/create` |
| 3 | Deployed the aligned model | `POST /models/deploy-model/{id}` |
| 4 | Chunked the document (3 methods compared) | — (local Python) |
| 5 | Generated embeddings for each chunk set | `POST /inference/embeddings` |
| 6 | Enhanced user queries for better retrieval | `POST /inference/chat/completions` |
| 7 | Retrieved candidate chunks via cosine similarity | `POST /inference/embeddings` |
| 8 | Re-ranked chunks for precision | `POST /inference/reranker` |
| 9 | Generated answers with the aligned model | `POST /inference/chat/completions` |

**Key takeaways:**
- Alignment is a single API call — Nugen handles the training infrastructure
- Embeddings + cosine similarity give us a lightweight vector search (no database needed)
- **Multiple chunking methods** let you tune retrieval granularity for your document type
- **Query enhancement** rewrites vague questions into specific ones, improving retrieval accuracy
- **Re-ranking** with Nugen's reranker endpoint adds a precision layer on top of embedding similarity
- RAG grounds the model's answers in actual document content, reducing hallucination
- The aligned model can answer domain questions more accurately than the base model

---

## Cleanup

Undeploy the model when you're done to free resources:

In [ ]:
# Uncomment to undeploy the model
# r = requests.delete(
#     f"{BASE_URL}/models/undeploy-model/{aligned_model_id}",
#     headers=HEADERS,
# )
# print(f"Undeploy: {r.status_code}")